Model: Random Forest Regressor predicting driver standings points

In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    explained_variance_score,
)
from sklearn.preprocessing import LabelEncoder
import warnings
import os
 
warnings.filterwarnings("ignore")

1. LOAD & PREPARE DATA

In [0]:
def load_data():
    """
    Load F1 data from Databricks/S3.
    Adjust paths to match your cluster's mount point.
    """
    # ── Primary tables ──────────────────────────────────────────────────────
    results    = spark.read.csv("dbfs:/Volumes/gr5069/raw/f1_data/results.csv",
                                header=True, inferSchema=True)
    drivers    = spark.read.csv("dbfs:/Volumes/gr5069/raw/f1_data/drivers.csv",
                                header=True, inferSchema=True)
    races      = spark.read.csv("dbfs:/Volumes/gr5069/raw/f1_data/races.csv",
                                header=True, inferSchema=True)
    constructors = spark.read.csv("dbfs:/Volumes/gr5069/raw/f1_data/constructors.csv",
                                  header=True, inferSchema=True)
 
    # Convert to pandas
    results_pd      = results.toPandas()
    drivers_pd      = drivers.toPandas()
    races_pd        = races.toPandas()
    constructors_pd = constructors.toPandas()
 
    # ── Merge ────────────────────────────────────────────────────────────────
    df = (results_pd
          .merge(drivers_pd[["driverId", "driverRef", "nationality",
                              "dob", "number"]],
                 on="driverId", how="left")
          .merge(races_pd[["raceId", "year", "round", "circuitId",
                            "name"]],
                 on="raceId", how="left")
          .merge(constructors_pd[["constructorId", "constructorRef",
                                   "nationality"]],
                 on="constructorId", how="left",
                 suffixes=("_driver", "_constructor")))
    return df
 
 
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Feature engineering on the merged F1 DataFrame."""

    # The CSV column is 'rank', rename to 'fastestLapRank' for clarity
    if "rank" in df.columns and "fastestLapRank" not in df.columns:
        df = df.rename(columns={"rank": "fastestLapRank"})

    # Numeric coercion
    for col in ["grid", "laps", "milliseconds", "fastestLapSpeed",
                "fastestLapRank", "points"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
 
    df = df.dropna(subset=["points", "grid", "laps"])
    df = df[df["points"] >= 0]
 
    # Encode categoricals
    le = LabelEncoder()
    for cat in ["driverRef", "constructorRef",
                "nationality_driver", "nationality_constructor"]:
        if cat in df.columns:
            df[cat + "_enc"] = le.fit_transform(df[cat].astype(str))
 
    # Driver age at race
    df["dob"]       = pd.to_datetime(df["dob"], errors="coerce")
    df["race_date"] = pd.to_datetime(df.get("date", pd.NaT), errors="coerce")
    df["driver_age"] = (
        (df["race_date"] - df["dob"]).dt.days / 365.25
    ).fillna(df["dob"].notna().mean())   # fill with mean age where missing
 
    # Derived numerics
    df["lap_time_s"] = df["milliseconds"] / (df["laps"] * 1000 + 1e-9)
    df["fastestLapSpeed"] = df["fastestLapSpeed"].fillna(
        df["fastestLapSpeed"].median()
    )
    df["fastestLapRank"] = df["fastestLapRank"].fillna(20)
 
    return df
 
 
FEATURE_COLS = [
    "grid", "laps", "fastestLapSpeed", "fastestLapRank",
    "round", "year", "circuitId",
    "driverRef_enc", "constructorRef_enc",
    "nationality_driver_enc", "nationality_constructor_enc",
    "driver_age", "lap_time_s",
]
TARGET = "points"

2. ARTIFACT HELPERS

In [0]:
def plot_residuals(y_test, y_pred, run_name: str) -> str:
    """Residual plot → PNG."""
    residuals = y_test - y_pred
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
 
    axes[0].scatter(y_pred, residuals, alpha=0.4, color="steelblue")
    axes[0].axhline(0, color="red", linestyle="--")
    axes[0].set_xlabel("Predicted Points")
    axes[0].set_ylabel("Residuals")
    axes[0].set_title(f"Residuals vs Predicted – {run_name}")
 
    axes[1].hist(residuals, bins=40, color="steelblue", edgecolor="white")
    axes[1].set_xlabel("Residual")
    axes[1].set_title("Residual Distribution")
 
    plt.tight_layout()
    path = f"/tmp/residuals_{run_name}.png"
    plt.savefig(path, dpi=100)
    plt.close()
    return path
 
 
def plot_feature_importance(model, feature_names: list, run_name: str) -> str:
    """Feature importance bar chart → PNG."""
    importances = model.feature_importances_
    idx = np.argsort(importances)[::-1]
 
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(range(len(importances)), importances[idx], color="steelblue")
    ax.set_xticks(range(len(importances)))
    ax.set_xticklabels([feature_names[i] for i in idx], rotation=45, ha="right")
    ax.set_title(f"Feature Importance – {run_name}")
    ax.set_ylabel("Importance")
    plt.tight_layout()
 
    path = f"/tmp/feature_importance_{run_name}.png"
    plt.savefig(path, dpi=100)
    plt.close()
    return path
 
 
def save_feature_importance_csv(model, feature_names: list, run_name: str) -> str:
    """Feature importance → CSV artifact."""
    df_fi = pd.DataFrame({
        "feature":    feature_names,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False)
 
    path = f"/tmp/feature_importance_{run_name}.csv"
    df_fi.to_csv(path, index=False)
    return path

3. SINGLE TRAINING RUN

In [0]:
def run_experiment(
    X_train, X_test, y_train, y_test,
    feature_names: list,
    run_name: str,
    n_estimators: int  = 100,
    max_depth:    int | None = None,
    min_samples_split: int  = 2,
    min_samples_leaf:  int  = 1,
    max_features: str  = "sqrt",
    random_state: int  = 42,
):
    with mlflow.start_run(run_name=run_name):
 
        # ── Train ────────────────────────────────────────────────────────────
        model = RandomForestRegressor(
            n_estimators       = n_estimators,
            max_depth          = max_depth,
            min_samples_split  = min_samples_split,
            min_samples_leaf   = min_samples_leaf,
            max_features       = max_features,
            random_state       = random_state,
            n_jobs             = -1,
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
 
        # ── Metrics ──────────────────────────────────────────────────────────
        mae  = mean_absolute_error(y_test, y_pred)
        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)
        evs  = explained_variance_score(y_test, y_pred)
        cv_r2 = cross_val_score(model, X_train, y_train,
                                cv=5, scoring="r2").mean()
 
        # ── Log params ───────────────────────────────────────────────────────
        mlflow.log_params({
            "n_estimators":      n_estimators,
            "max_depth":         max_depth if max_depth else "None",
            "min_samples_split": min_samples_split,
            "min_samples_leaf":  min_samples_leaf,
            "max_features":      max_features,
            "random_state":      random_state,
            "model":             "RandomForestRegressor",
        })
 
        # ── Log metrics ──────────────────────────────────────────────────────
        mlflow.log_metrics({
            "mae":    mae,
            "mse":    mse,
            "rmse":   rmse,
            "r2":     r2,
            "explained_variance": evs,
            "cv_r2":  cv_r2,
        })
 
        # ── Log model ────────────────────────────────────────────────────────
        mlflow.sklearn.log_model(model, "random-forest-model")
 
        # ── Log artifacts ────────────────────────────────────────────────────
        res_path = plot_residuals(y_test, y_pred, run_name)
        fi_plot  = plot_feature_importance(model, feature_names, run_name)
        fi_csv   = save_feature_importance_csv(model, feature_names, run_name)
 
        mlflow.log_artifact(res_path, artifact_path="residuals.png")
        mlflow.log_artifact(fi_plot,  artifact_path="feature_importance.png")
        mlflow.log_artifact(fi_csv,   artifact_path="feature-importance.csv")
 
        print(
            f"[{run_name}] n_est={n_estimators} max_depth={max_depth} "
            f"→ MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.4f}  CV-R²={cv_r2:.4f}"
        )
 
        return {
            "run_name": run_name, "mae": mae, "mse": mse,
            "rmse": rmse, "r2": r2, "cv_r2": cv_r2,
        }

4. EXPERIMENT GRID  (≥10 runs)

In [0]:
EXPERIMENT_GRID = [
    # run_name,               n_est,  max_depth, min_samp_split, min_leaf, max_feat
    ("Run01_baseline",          100,   None,       2,  1, "sqrt"),
    ("Run02_shallow",           100,    5,          2,  1, "sqrt"),
    ("Run03_medium_depth",      200,   10,          2,  1, "sqrt"),
    ("Run04_deep",              300,   20,          2,  1, "sqrt"),
    ("Run05_many_trees",        500,   None,        2,  1, "sqrt"),
    ("Run06_large_min_split",   200,   10,         10,  5, "sqrt"),
    ("Run07_log2_features",     200,   10,          2,  1, "log2"),
    ("Run08_all_features",      200,   10,          2,  1, 1.0),
    ("Run09_high_leaf",         300,   15,          5, 10, "sqrt"),
    ("Run10_balanced",          400,   12,          4,  2, "sqrt"),
    ("Run11_very_deep",         500,   30,          2,  1, "sqrt"),
    ("Run12_small_fast",         50,    8,          5,  3, "log2"),
]

5. MAIN

In [0]:
def main():
    # ── Set experiment ───────────────────────────────────────────────────────
    mlflow.set_experiment("/Users/sz3388@columbia.edu/hw4_f1_rf")

    # ── Load & prepare data ──────────────────────────────────────────────────
    print("Loading data …")
    df = engineer_features(load_data())

    available = [c for c in FEATURE_COLS if c in df.columns]
    X = df[available].fillna(0)
    y = df[TARGET]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    print(f"Train: {X_train.shape}  Test: {X_test.shape}")

    # ── Run experiments ──────────────────────────────────────────────────────
    results = []
    for (run_name, n_est, max_dep, mss, msl, mf) in EXPERIMENT_GRID:
        r = run_experiment(
            X_train, X_test, y_train, y_test,
            feature_names    = available,
            run_name         = run_name,
            n_estimators     = n_est,
            max_depth        = max_dep,
            min_samples_split= mss,
            min_samples_leaf = msl,
            max_features     = mf,
        )
        results.append(r)

    # ── Summary table ────────────────────────────────────────────────────────
    summary = pd.DataFrame(results).sort_values("r2", ascending=False)
    print("\n" + "="*70)
    print("EXPERIMENT SUMMARY  (sorted by R²)")
    print("="*70)
    print(summary.to_string(index=False))

    best = summary.iloc[0]
    print(f"\n✅ Best Run: {best['run_name']}"
          f"  R²={best['r2']:.4f}  RMSE={best['rmse']:.3f}")


if __name__ == "__main__":
    main()